# 🤖 AI Interview Coach
### No API Key Required — Runs 100% Free on Google Colab

This chatbot will:
- **Ask you interview questions** based on your chosen role
- **Evaluate your answers** and give detailed feedback
- **Score your responses** out of 10
- **Give tips** to improve weak answers

> ⚡ **Tip:** Go to `Runtime > Change runtime type > T4 GPU` for faster responses!

In [1]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate torch gradio sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.3 MB/s eta 0:00:00


In [2]:
# Cell 2: Import libraries
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import random
import re

print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [3]:
# Cell 3: Load FREE model (no API key needed)
# Options:
# MODEL_ID = 'google/flan-t5-large'                     # Very fast, CPU-friendly
# MODEL_ID = 'microsoft/phi-2'                          # High quality, needs more RAM
# MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.1'       # Best quality, needs A100
MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # Default: fast and free

print(f'Loading model: {MODEL_ID}')
print('This may take 1-3 minutes on first run...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)

generator = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=400,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print('Model loaded and ready!')

Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
This may take 1-3 minutes on first run...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded and ready!


In [4]:
# Cell 4: Interview Question Bank
QUESTION_BANK = {
    'Python Developer': [
        'Explain the difference between a list and a tuple in Python.',
        'What are Python decorators and how do you use them?',
        'Explain the concept of generators in Python with an example.',
        'What is the difference between deep copy and shallow copy?',
        'How does Python GIL affect multithreading?',
        'Explain the difference between *args and **kwargs.',
        'What are context managers and how do you create one?',
        'Describe Python memory management and garbage collection.',
        'What is list comprehension and when should you use it?',
        'Explain OOP concepts in Python with examples.'
    ],
    'Data Science': [
        'What is the difference between supervised and unsupervised learning?',
        'Explain overfitting and how to prevent it.',
        'What is cross-validation and why is it important?',
        'Explain the bias-variance tradeoff.',
        'What are the steps in a typical machine learning pipeline?',
        'How do you handle missing data in a dataset?',
        'Explain the difference between classification and regression.',
        'What is feature engineering and why does it matter?',
        'Explain how a Random Forest algorithm works.',
        'What is the ROC curve and AUC score?'
    ],
    'Web Developer': [
        'Explain the difference between HTTP and HTTPS.',
        'What is REST API and what are its principles?',
        'Explain the event loop in JavaScript.',
        'What is the difference between SQL and NoSQL databases?',
        'How does browser caching work?',
        'Explain the concept of responsive design.',
        'What is CORS and why does it exist?',
        'Explain how OAuth2 authentication works.',
        'What are WebSockets and when would you use them?',
        'Describe the MVC design pattern.'
    ],
    'HR / Behavioral': [
        'Tell me about yourself and your background.',
        'Describe a situation where you handled a conflict with a teammate.',
        'What is your greatest strength and weakness?',
        'Tell me about a time you failed and what you learned from it.',
        'Why do you want to work for our company?',
        'Describe a time when you had to meet a tight deadline.',
        'How do you prioritize tasks when you have multiple deadlines?',
        'Tell me about a time you showed leadership.',
        'Where do you see yourself in 5 years?',
        'Why are you leaving your current job?'
    ],
    'Machine Learning Engineer': [
        'Explain the difference between batch and online learning.',
        'How do you deploy a machine learning model to production?',
        'What is model drift and how do you monitor it?',
        'Explain the transformer architecture briefly.',
        'What is transfer learning and when is it useful?',
        'Describe how you would design an A/B testing framework.',
        'What are embeddings and how are they used in NLP?',
        'Explain gradient descent and its variants.',
        'How do you evaluate a recommendation system?',
        'What is regularization and why is it needed?'
    ]
}

print('Question bank loaded!')
print('Roles:', ', '.join(QUESTION_BANK.keys()))

Question bank loaded!
Roles: Python Developer, Data Science, Web Developer, HR / Behavioral, Machine Learning Engineer


In [5]:
# Cell 5: Core chatbot logic

def generate_response(prompt, max_tokens=400):
    try:
        output = generator(prompt, max_new_tokens=max_tokens)
        full_text = output[0]['generated_text']
        response = full_text[len(prompt):].strip()
        return response
    except Exception as e:
        return f'[Model error: {str(e)}]'


def get_next_question(role, used_questions):
    all_questions = QUESTION_BANK.get(role, QUESTION_BANK['HR / Behavioral'])
    remaining = [q for q in all_questions if q not in used_questions]
    if not remaining:
        return None
    return random.choice(remaining)


def evaluate_answer(question, answer, role):
    prompt = (
        f'<|system|>\nYou are an expert technical interviewer for {role} positions.'
        f' Evaluate the answer carefully.\n<|user|>\n'
        f'Interview Question: {question}\n\n'
        f'Candidate Answer: {answer}\n\n'
        f'Evaluate this answer with:\n'
        f'1. SCORE: X/10\n'
        f'2. STRENGTHS: What the candidate did well\n'
        f'3. WEAKNESSES: What was missing or incorrect\n'
        f'4. IDEAL ANSWER: Key points a strong answer should include\n'
        f'5. TIP: One specific improvement suggestion\n'
        f'<|assistant|>\n'
    )
    return generate_response(prompt, max_tokens=450)


def new_state():
    return {
        'phase': 'role_selection',
        'role': None,
        'used_questions': [],
        'current_question': None,
        'score_total': 0,
        'questions_done': 0,
        'max_questions': 5
    }


def chat(user_message, history, state):
    if state is None:
        state = new_state()

    response = ''
    phase = state['phase']
    roles = list(QUESTION_BANK.keys())

    if phase == 'role_selection':
        selected_role = None
        msg = user_message.strip()
        if msg.isdigit():
            idx = int(msg) - 1
            if 0 <= idx < len(roles):
                selected_role = roles[idx]
        else:
            for role in roles:
                if msg.lower() in role.lower() or role.lower() in msg.lower():
                    selected_role = role
                    break

        if selected_role:
            state['role'] = selected_role
            state['phase'] = 'interviewing'
            question = get_next_question(selected_role, state['used_questions'])
            state['current_question'] = question
            state['used_questions'].append(question)
            response = (
                f'Great! Starting your **{selected_role}** interview.\n\n'
                f'I will ask you {state["max_questions"]} questions and give feedback after each.\n\n'
                f'---\n\n**Question 1/{state["max_questions"]}:**\n\n'
                f'{question}\n\nTake your time and type your answer below.'
            )
        else:
            role_list = '\n'.join([f'{i+1}. {r}' for i, r in enumerate(roles)])
            response = f'Please type a number or role name:\n{role_list}'

    elif phase == 'interviewing':
        answer = user_message.strip()
        if len(answer) < 10:
            response = 'Please give a more detailed answer (at least a sentence or two).'
        else:
            state['questions_done'] += 1
            q_num = state['questions_done']
            feedback = evaluate_answer(state['current_question'], answer, state['role'])
            score_match = re.search(r'(\d+)\s*/\s*10', feedback)
            if score_match:
                state['score_total'] += int(score_match.group(1))

            response = f'**Feedback for Question {q_num}:**\n\n{feedback}\n\n---\n'

            if state['questions_done'] < state['max_questions']:
                next_q = get_next_question(state['role'], state['used_questions'])
                if next_q:
                    state['current_question'] = next_q
                    state['used_questions'].append(next_q)
                    response += (
                        f'**Question {q_num + 1}/{state["max_questions"]}:**\n\n'
                        f'{next_q}\n\nType your answer below.'
                    )
                else:
                    state['phase'] = 'completed'
                    response += _summary(state)
            else:
                state['phase'] = 'completed'
                response += _summary(state)

    elif phase == 'completed':
        if 'restart' in user_message.lower():
            state = new_state()
            role_list = '\n'.join([f'{i+1}. {r}' for i, r in enumerate(roles)])
            response = f'New Interview Session!\n\nChoose your role:\n{role_list}'
        else:
            response = 'Interview complete. Type **restart** to begin a new session.'

    history.append((user_message, response))
    return history, state


def _summary(state):
    done = state['questions_done']
    total = state['score_total']
    avg = total / max(done, 1)
    if avg >= 8:
        verdict = 'Excellent performance! You are well-prepared!'
    elif avg >= 6:
        verdict = 'Good effort! Review weak areas and keep practicing.'
    else:
        verdict = 'Keep studying - review the ideal answers above and try again.'
    return (
        f'**Interview Complete!**\n\n'
        f'**Final Score: {total}/{done * 10}** (Average: {avg:.1f}/10)\n\n'
        f'{verdict}\n\nType **restart** to start a new interview.'
    )


def start_session():
    state = new_state()
    roles = list(QUESTION_BANK.keys())
    role_list = '\n'.join([f'{i+1}. {r}' for i, r in enumerate(roles)])
    welcome = (
        'Welcome to the AI Interview Coach!\n\n'
        'I will ask you 5 interview questions and give detailed feedback on each answer.\n\n'
        '**Choose your interview role by typing the number:**\n\n'
        + role_list
    )
    history = [('', welcome)]
    return history, state


print('Chatbot logic ready!')

Chatbot logic ready!


In [6]:
# Cell 6: Launch Gradio UI

css = """
.gradio-container { max-width: 860px; margin: auto; }
footer { display: none !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(), title='AI Interview Coach', css=css) as demo:

    gr.HTML("""
        <div style='text-align:center; padding: 10px 0'>
            <h1 style='font-size:2em; margin:0'>AI Interview Coach</h1>
            <p style='color:#666; margin:4px 0'>Practice interviews with instant AI feedback | No API key needed</p>
        </div>
    """)

    state = gr.State(None)

    chatbot = gr.Chatbot(label='Interview Session', height=500, bubble_full_width=False)

    with gr.Row():
        msg = gr.Textbox(
            placeholder='Type your answer here and press Enter...',
            label='', scale=5, lines=2
        )
        send_btn = gr.Button('Send', scale=1, variant='primary')

    with gr.Row():
        restart_btn = gr.Button('New Interview', scale=1)
        clear_btn = gr.Button('Clear Chat', scale=1)

    with gr.Accordion('How to Use', open=False):
        gr.Markdown("""
        **Steps:**
        1. Type the number of your chosen interview role (1-5)
        2. Answer each question in the text box
        3. Receive AI feedback with score, strengths, weaknesses, and tips
        4. After 5 questions, see your final score
        5. Type **restart** or click New Interview to try again

        **Available Roles:** Python Developer | Data Science | Web Developer | ML Engineer | HR/Behavioral
        """)

    def respond(message, history, state):
        if not message.strip():
            return history, '', state
        history, state = chat(message, history, state)
        return history, '', state

    def restart(state):
        history, state = start_session()
        return history, '', state

    msg.submit(respond, [msg, chatbot, state], [chatbot, msg, state])
    send_btn.click(respond, [msg, chatbot, state], [chatbot, msg, state])
    restart_btn.click(restart, [state], [chatbot, msg, state])
    clear_btn.click(lambda: ([], ''), outputs=[chatbot, msg])
    demo.load(restart, [state], [chatbot, msg, state])

demo.launch(share=True, debug=False, quiet=True)

/tmp/ipykernel_1389/798382949.py:8: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='AI Interview Coach', css=css) as demo:
/tmp/ipykernel_1389/798382949.py:8: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='AI Interview Coach', css=css) as demo:
/tmp/ipykernel_1389/798382949.py:19: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label='Interview Session', height=500, bubble_full_width=False)
/tmp/ipykernel_

* Running on public URL: https://355dd489e0e31199d1.gradio.live


---
## Customization Tips

**Change number of questions** — in Cell 5, find `new_state()` and change:
```python
'max_questions': 10
```

**Add your own role/questions** — in Cell 4, add to `QUESTION_BANK`:
```python
QUESTION_BANK['DevOps Engineer'] = [
    'What is CI/CD and how does it work?',
    'Explain Docker vs Kubernetes.',
]
```

**Use a better model** — in Cell 3 (needs more GPU RAM):
```python
MODEL_ID = 'microsoft/phi-2'   # ~5GB, better quality
```